In [0]:
-- LA PHASE GOLD_LAYER --

-- Databricks notebook source
SELECT * FROM silver_db.processed_trips LIMIT 10;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timestamp,file_name
2,2024-04-01T00:41:12Z,2024-04-01T00:55:29Z,1,5.6,1,N,264,264,1,25.4,1.0,0.5,10.0,0.0,1.0,37.9,0.0,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:48:42Z,2024-04-01T01:05:30Z,1,3.55,1,N,186,236,1,20.5,1.0,0.5,5.1,0.0,1.0,30.6,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
1,2024-04-01T00:08:32Z,2024-04-01T00:10:24Z,1,0.7,1,N,236,263,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
1,2024-04-01T00:02:09Z,2024-04-01T00:20:33Z,1,4.7,1,N,138,146,1,21.9,7.75,0.5,7.8,0.0,1.0,38.95,0.0,1.75,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:27:16Z,2024-04-01T00:43:44Z,2,3.24,1,N,186,87,1,18.4,1.0,0.5,4.68,0.0,1.0,28.08,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
1,2024-04-01T00:41:26Z,2024-04-01T01:25:31Z,1,21.5,2,N,132,143,1,70.0,4.25,0.5,16.54,6.94,1.0,99.23,2.5,1.75,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:19:04Z,2024-04-01T00:29:25Z,3,2.39,1,N,249,230,1,13.5,1.0,0.5,3.7,0.0,1.0,22.2,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:35:37Z,2024-04-01T00:41:16Z,4,1.43,1,N,100,113,1,8.6,1.0,0.5,2.04,0.0,1.0,15.64,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:23:35Z,2024-04-01T00:34:58Z,1,2.65,1,N,161,239,1,14.2,1.0,0.5,3.84,0.0,1.0,23.04,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:33:47Z,2024-04-01T00:38:40Z,1,1.51,1,N,137,141,1,8.6,1.0,0.5,2.72,0.0,1.0,16.32,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet


In [0]:
CREATE DATABASE IF NOT EXISTS gold_db;

USE gold_db;

SELECT current_database();

current_schema()
gold_db


In [0]:
SELECT * FROM bronze_db.taxi_zones LIMIT 10;

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
CREATE OR REPLACE TABLE processed_trips_with_zones 
USING DELTA 
AS
SELECT
  s.*,
  pu_zone.Borough AS pickup_borough,
  do_zone.Borough AS dropoff_borough,
  pu_zone.Zone AS pickup_zone,
  do_zone.Zone AS dropoff_zone,
  DATE(s.tpep_pickup_datetime) AS pickup_date,
  HOUR(s.tpep_pickup_datetime) AS pickup_hour,
  HOUR(s.tpep_dropoff_datetime) AS dropoff_hour,
  DAYOFWEEK(s.tpep_pickup_datetime) AS pickup_day_of_week
FROM silver_db.processed_trips s
LEFT JOIN bronze_db.taxi_zones pu_zone 
  ON s.PULocationID = pu_zone.LocationID
LEFT JOIN bronze_db.taxi_zones do_zone 
  ON s.DOLocationID = do_zone.LocationID

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) FROM processed_trips_with_zones;

count(1)
12292364


In [0]:
SELECT * FROM processed_trips_with_zones LIMIT 10;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timestamp,file_name,pickup_borough,dropoff_borough,pickup_zone,dropoff_zone,pickup_date,pickup_hour,dropoff_hour,pickup_day_of_week
1,2024-05-01T00:59:15Z,2024-05-01T01:23:50Z,1,6.1,1,N,138,145,1,28.2,7.75,0.5,5.0,0.0,1.0,42.45,0.0,1.75,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Queens,Queens,LaGuardia Airport,Long Island City/Hunters Point,2024-05-01,0,1,4
2,2024-04-30T23:58:26Z,2024-05-01T00:29:42Z,1,11.23,1,N,138,249,1,46.4,6.0,0.5,8.72,0.0,1.0,66.87,2.5,1.75,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Queens,Manhattan,LaGuardia Airport,West Village,2024-04-30,23,0,3
2,2024-05-01T00:57:17Z,2024-05-01T01:14:15Z,1,9.02,1,N,138,170,1,35.9,6.0,0.5,10.57,6.94,1.0,65.16,2.5,1.75,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Queens,Manhattan,LaGuardia Airport,Murray Hill,2024-05-01,0,1,4
2,2024-05-01T00:24:47Z,2024-05-01T00:48:51Z,1,6.53,1,N,87,133,1,30.3,1.0,0.5,7.06,0.0,1.0,42.36,2.5,0.0,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Manhattan,Brooklyn,Financial District North,Kensington,2024-05-01,0,0,4
2,2024-05-01T00:11:20Z,2024-05-01T00:52:10Z,1,14.38,1,N,161,165,1,61.8,1.0,0.5,0.0,0.0,1.0,66.8,2.5,0.0,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Manhattan,Brooklyn,Midtown Center,Midwood,2024-05-01,0,0,4
2,2024-05-01T00:54:41Z,2024-05-01T01:14:00Z,1,9.19,1,N,138,151,1,37.3,6.0,0.5,10.7,6.94,1.0,64.19,0.0,1.75,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Queens,Manhattan,LaGuardia Airport,Manhattan Valley,2024-05-01,0,1,4
2,2024-05-01T00:06:44Z,2024-05-01T00:27:44Z,1,4.52,1,N,186,33,1,23.3,1.0,0.5,5.09,0.0,1.0,33.39,2.5,0.0,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Manhattan,Brooklyn,Penn Station/Madison Sq West,Brooklyn Heights,2024-05-01,0,0,4
1,2024-05-01T00:11:37Z,2024-05-01T00:20:05Z,1,2.7,1,N,237,166,1,12.8,3.5,0.5,3.2,0.0,1.0,21.0,2.5,0.0,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Manhattan,Manhattan,Upper East Side South,Morningside Heights,2024-05-01,0,0,4
2,2024-05-01T00:05:24Z,2024-05-01T00:29:26Z,1,4.28,1,N,114,142,1,25.4,1.0,0.5,6.08,0.0,1.0,36.48,2.5,0.0,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Manhattan,Manhattan,Greenwich Village South,Lincoln Square East,2024-05-01,0,0,4
2,2024-05-01T00:55:12Z,2024-05-01T01:05:09Z,1,1.49,1,N,234,4,1,11.4,1.0,0.5,1.5,0.0,1.0,17.9,2.5,0.0,2025-11-13T19:28:08.839054Z,yellow_tripdata_2024_05.parquet,Manhattan,Manhattan,Union Sq,Alphabet City,2024-05-01,0,1,4


In [0]:
SELECT DISTINCT file_name FROM processed_trips_with_zones ORDER BY file_name;